Optimzing Model Parameters
When we have a model and data, we then have to train, validate and test our model by optimizing its parameterson our data.
Training a model is an iterative process;
1. In each iteration, the model guesses(predictions) the output;
2. Calculates the errors in its guess(loss);
3. Collects the derivatives of the error with respect to its parameters and then optimizes these parameters using GRADIENT DESCENT.

In [17]:
import torch

In [18]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [34]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self,x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


In [35]:
model = NeuralNetwork()

In [28]:
training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = v2.Compose(
        [v2.ToImage(), v2.ToDtype(torch.float32, scale = True)]
    )
)

train_dataloader = DataLoader(training_data, batch_size = 64)

testing_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = False,
    transform = v2.Compose(
        [v2.ToImage(), v2.ToDtype(torch.float32, scale = True)]
    )
)

test_dataloader = DataLoader(testing_data, batch_size=64)






Hyperparameters

These are adjustable parameters that let you control the model optimization process.
Different hyperparameter values can impact model training and convergence rates

for our use case we define the hyperparameters for training:
1. Learning rate = 0.001
2. batch_size = 64
3. epochs = 5


In [29]:
learning_rate = 1e-3
batch_size = 64
epochs = 5


<h3>Optimization Loop</h3>
Once our Hyperparameters are defined, we can train and optimize out model with an optimization loop.
Each Iteration of the optimization loop is called an Epoch.

Each Epoch has two components
1. The Train loop - Iterate over the training data, and try to converge to optimal parameters (weights)
2. The Validation/Test loop - Iterate over the test dataset to check if the model performance is improving.


<h3>Loss Function</h3>
Used to measure to degree of dissimilarity of obtained results of the target value.
It is the function we want to minimize during training. 

To calculate the loss we make prediction using the inputs of our given data sample and compare it against the true data label value.

Common loss functions include, 
nn.MSELoss --> Mean Square Error, for regression tasks,
nn.NLLLoss --> Negative log Likelihood for classification.
nn.CrossEntropyLoss combines, nn.LogSoftmax and nn.NLLLoss

We pass our model's output logits to nn.CrossEntropyLoss, which will normalize the logits and compute the prediction error

In [30]:
# loss_fn = nn.CrossEntropyLoss()

<h4>Optimizer</h4>
Optimization is the process of adjusting the model parameters to resuce tmodel error in each training step.
Optimization algorithms define how this process is performed.

All optimization logic is encapsulated in the optimizer object.
For our use case we use Stochastic Gradient Descent.

Other examples of optimizers are : 
1. ADAM
2. RMSProp

Each of them work better for different kind of model and data

In [31]:
# optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

Inside the training loop, optimization happens in three steps:

Call `optimizer.zero_grad()` to reset the gradients of model parameters. Gradients by default add up; to prevent double-counting, we explicitly zero them at each iteration.

Backpropagate the prediction loss with a call to `loss.backward()`. PyTorch deposits the gradients of the loss w.r.t. each parameter.

Once we have our gradients, we call `optimizer.step()` to adjust the parameters by the gradients collected in the backward pass.

In [32]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [36]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.302902  [   64/60000]
loss: 2.290094  [ 6464/60000]
loss: 2.281707  [12864/60000]
loss: 2.279628  [19264/60000]
loss: 2.251673  [25664/60000]
loss: 2.232075  [32064/60000]
loss: 2.226251  [38464/60000]
loss: 2.202875  [44864/60000]
loss: 2.202085  [51264/60000]
loss: 2.164951  [57664/60000]
Test Error: 
 Accuracy: 53.2%, Avg loss: 2.168098 

Epoch 2
-------------------------------
loss: 2.174006  [   64/60000]
loss: 2.163317  [ 6464/60000]
loss: 2.121034  [12864/60000]
loss: 2.136600  [19264/60000]
loss: 2.076379  [25664/60000]
loss: 2.030627  [32064/60000]
loss: 2.042846  [38464/60000]
loss: 1.978171  [44864/60000]
loss: 1.982674  [51264/60000]
loss: 1.901855  [57664/60000]
Test Error: 
 Accuracy: 59.3%, Avg loss: 1.912515 

Epoch 3
-------------------------------
loss: 1.938475  [   64/60000]
loss: 1.907816  [ 6464/60000]
loss: 1.810439  [12864/60000]
loss: 1.846583  [19264/60000]
loss: 1.727015  [25664/60000]
loss: 1.685459  [32064/600